# NOT IN en SQL : l'erreur silencieuse des valeurs NULL

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/MonShotData/blob/main/SQL/Clause-NOT-IN.ipynb)

📖 **Le shot complet** : [NOT IN en SQL : l'erreur invisible qui te coûte des heures](https://www.monshotdata.com/p/erreur-silencieuse-sql-not-in-null)

---

Une requête `NOT IN` qui marche très bien... jusqu'au jour où une valeur `NULL` se glisse dans la sous-requête : elle ne renvoie alors **plus aucune ligne**, sans la moindre erreur. Ici tu reproduis le piège avec DuckDB, tu comprends pourquoi `IN` n'a pas ce problème, et tu vois trois façons de t'en protéger.

**À la fin de ce notebook, tu sauras :**

- pourquoi `NOT IN` ne renvoie plus rien dès que la sous-requête contient un `NULL`, alors que `IN` fonctionne normalement ;
- comment SQL raisonne avec trois valeurs de vérité (`TRUE`, `FALSE` et `UNKNOWN`) ;
- trois façons d'éviter le piège : filtrer les `NULL`, utiliser un anti join, utiliser `NOT EXISTS`.

### Avant de lancer

- **Installation** : la première cellule installe **DuckDB** (quelques secondes). C'est un moteur SQL qui tourne dans le notebook, sans serveur ni base à créer.
- **Les données** : les deux petites tables sont créées dans le notebook avec Pandas. DuckDB lit les DataFrames directement par leur nom, comme s'ils étaient des tables.

🧪 *Testé le 22 novembre 2026 : exécution complète de haut en bas (Python 3.13, DuckDB 2.0, pandas 3.0 et NumPy 2.5). Les sorties affichées viennent de cette exécution.*

## 1. Mise en place

In [1]:
%pip install -q duckdb

In [2]:
import duckdb
import numpy as np
import pandas as pd

## 2. Le même piège existe en NumPy

Les valeurs manquantes sont une source classique d'erreurs inattendues. Exemple avec un array NumPy contenant un `NaN` :

In [3]:
x = np.array([1, 2, 3, np.nan])

print("Somme avec np.sum    :", np.sum(x))      # nan : un seul NaN contamine tout le résultat
print("Somme avec np.nansum :", np.nansum(x))   # 6.0 : on ignore explicitement les valeurs manquantes

Somme avec np.sum    : nan
Somme avec np.nansum : 6.0


On s'attendrait à 6, NumPy renvoie `nan`. En SQL, une erreur du même genre se cache dans la clause `NOT IN`.

## 3. Les données

Une table `students` (cinq étudiants) et une table `names` (prénoms à exclure), pour l'instant **sans valeur manquante**.

In [4]:
students = pd.DataFrame({
    "roll_no": [101, 102, 103, 104, 105],
    "first_name": ["John", "Emma", "Michael", "Sophia", "Ethan"],
    "last_name": ["Doe", "Smith", "Johnson", "Brown", "Williams"],
    "grade": ["A", "B", "C", "A", "B"],
})
students

,roll_no,first_name,last_name,grade
0,101,John,Doe,A
1,102,Emma,Smith,B
2,103,Michael,Johnson,C
3,104,Sophia,Brown,A
4,105,Ethan,Williams,B


In [5]:
names = pd.DataFrame({"first_name": ["John", "Peter", "Ethan"]})
names

,first_name
0,John
1,Peter
2,Ethan


L'objectif : sélectionner les étudiants dont le prénom **n'est pas** dans `names`.

In [6]:
query = """
SELECT *
FROM students
WHERE first_name NOT IN (
    SELECT first_name
    FROM names
);
"""

duckdb.sql(query).df()

,roll_no,first_name,last_name,grade
0,102,Emma,Smith,B
1,103,Michael,Johnson,C
2,104,Sophia,Brown,A


Résultat attendu : Emma, Michael et Sophia. Tout va bien jusque-là.

## 4. Ajoutons un `NULL` dans `names`

In [7]:
names = pd.DataFrame({"first_name": [None, "John", "Peter", "Ethan"]})
names

,first_name
0,NaN
1,John
2,Peter
3,Ethan


On relance **exactement** la même requête :

In [8]:
query = """
SELECT *
FROM students
WHERE first_name NOT IN (SELECT first_name FROM names);
"""

duckdb.sql(query).df()

,roll_no,first_name,last_name,grade


Aucun résultat, et aucune erreur. Et pourtant, avec `IN` (le contraire), tout continue de fonctionner :

In [9]:
query = """
SELECT *
FROM students
WHERE first_name IN (SELECT first_name FROM names);
"""

duckdb.sql(query).df()

,roll_no,first_name,last_name,grade
0,101,John,Doe,A
1,105,Ethan,Williams,B


## 5. Pourquoi `NOT IN` ne renvoie rien ?

La clause `WHERE` n'a besoin que d'une valeur booléenne par ligne. SQL développe `IN` et `NOT IN` ainsi :

- `x IN (a, b, c)` devient `x = a OR x = b OR x = c`. Il suffit qu'**une** condition soit `TRUE`.
- `x NOT IN (a, b, c)` devient `x != a AND x != b AND x != c`. Il faut que **toutes** les conditions soient `TRUE`.

Or comparer quoi que ce soit à `NULL` ne donne ni `TRUE` ni `FALSE`, mais **`UNKNOWN`** (en SQL, `NULL` signifie « valeur inconnue »). Donc `'Emma' != NULL` vaut `UNKNOWN`, et l'expression entière devient `UNKNOWN` : `WHERE` ne garde que les lignes `TRUE`, aucune ligne n'est retenue.

Vérifions-le sur trois expressions (`NULL` s'affiche `<NA>` ou `None`) :

In [10]:
duckdb.sql("""
SELECT
    'Emma' NOT IN ('John', 'Peter')          AS not_in_sans_null,   -- TRUE
    'Emma' NOT IN ('John', 'Peter', NULL)    AS not_in_avec_null,   -- NULL (= UNKNOWN)
    'John' IN     ('John', 'Peter', NULL)    AS in_avec_null        -- TRUE : un seul TRUE suffit avec OR
""").df()

,not_in_sans_null,not_in_avec_null,in_avec_null
0,True,<NA>,True


Avec `IN`, un seul `TRUE` suffit (`OR`), donc le `NULL` ne gêne pas. Avec `NOT IN`, le `NULL` rend l'expression inconnue pour **tous** les étudiants.

## 6. Trois solutions

L'objectif reste le même : les étudiants dont le prénom n'est pas dans `names`, qui contient maintenant un `NULL`.

### 6.1 Filtrer les `NULL` dans la sous-requête

In [11]:
query = """
SELECT *
FROM students
WHERE first_name NOT IN (
    SELECT first_name
    FROM names
    WHERE first_name IS NOT NULL
);
"""

duckdb.sql(query).df()

,roll_no,first_name,last_name,grade
0,102,Emma,Smith,B
1,103,Michael,Johnson,C
2,104,Sophia,Brown,A


### 6.2 Utiliser un anti join

Un **anti join** garde uniquement les lignes de la table de gauche qui n'ont **aucune correspondance** dans la table de droite : c'est exactement le besoin. Une comparaison avec `NULL` n'est jamais une correspondance, donc le `NULL` est simplement ignoré.

In [12]:
query = """
SELECT *
FROM students s
ANTI JOIN names n
    ON s.first_name = n.first_name;
"""

duckdb.sql(query).df()

,roll_no,first_name,last_name,grade
0,102,Emma,Smith,B
1,103,Michael,Johnson,C
2,104,Sophia,Brown,A


### 6.3 Utiliser `NOT EXISTS`

La forme standard du même raisonnement, disponible dans tous les SQL (l'écriture `ANTI JOIN` est propre à DuckDB et à quelques autres moteurs) :

In [13]:
query = """
SELECT *
FROM students s
WHERE NOT EXISTS (
    SELECT 1
    FROM names n
    WHERE n.first_name = s.first_name
);
"""

duckdb.sql(query).df()

,roll_no,first_name,last_name,grade
0,102,Emma,Smith,B
1,103,Michael,Johnson,C
2,104,Sophia,Brown,A


Les trois requêtes renvoient Emma, Michael et Sophia. Dans les trois cas, le `NULL` ne peut plus fausser le résultat.

## 7. Ce qu'il faut retenir

- `NOT IN` renvoie **zéro ligne** dès que la sous-requête contient un `NULL`, sans erreur ni avertissement ; `IN` n'a pas ce problème.
- SQL a trois valeurs de vérité : `TRUE`, `FALSE` et `UNKNOWN`. Comparer à `NULL` donne `UNKNOWN`, et `WHERE` ne garde que les lignes `TRUE`.
- `x NOT IN (...)` est un grand `AND` de différences : un seul `UNKNOWN` suffit à tout bloquer.
- Protège-toi avec `WHERE col IS NOT NULL` dans la sous-requête, un `ANTI JOIN` ou un `NOT EXISTS`.
- Méfie-toi surtout quand la colonne de la sous-requête peut contenir des `NULL` : l'erreur peut n'apparaître que le jour où une donnée manquante arrive.

---

## Pour aller plus loin

- [Semi, Anti et Natural Join : 3 joins SQL méconnus](https://www.monshotdata.com/p/semi-join-anti-join-natural-join-sql) — les joins utilisés ici
- [SQL GROUPING SETS : la technique sous-estimée pour agréger en un scan](https://www.monshotdata.com/p/sql-grouping-sets-agregations-multiples-scan-unique) — plusieurs agrégations en une seule requête
- [Tous les notebooks des shots](https://github.com/moncoachdata/MonShotData) — le code Python de la newsletter, rangé par thème